# Notebook 15: Structured Decoding

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebooks 02 (Transformer Architecture), 03 (Sampling Strategies), 04 (Decoding Algorithms)

---

## Background: The Reliability Problem

Language models are probabilistic text generators. When you ask them to output JSON,
they *usually* do — but not always. They might:
- Add extra commentary around the JSON
- Forget a closing brace
- Use a key name slightly different from your schema
- Output valid JSON with the wrong types

For interactive chat, this is annoying but recoverable. For automated pipelines that
parse the output programmatically, it's catastrophic. A single malformed response
can crash a data pipeline or leave a user facing an error.

The naive solution is prompt engineering: "Always output valid JSON. Only output JSON,
no other text." This works most of the time with powerful models at low temperature.
But "most of the time" isn't good enough for production systems that need 99.9%+ reliability.

### The Structured Decoding Insight

Instead of asking the model to constrain itself through natural language instructions,
**enforce the constraint mathematically at the token-sampling step**.

At every decode step, before sampling a token:
1. Look at what tokens have been generated so far
2. Determine which tokens are *valid continuations* given the output schema
3. Zero out (or set to -infinity) the logits for all *invalid* tokens
4. Sample from the remaining distribution

The result: **100% structurally valid output, guaranteed**, while still sampling from the
model's probability distribution over valid completions. You don't sacrifice quality —
the model still picks the most likely valid completion. You just remove the possibility
of structural errors.

### The State Machine Approach

To know which tokens are valid at each step, you need to know the *state* of the output
parsing. This is exactly what a finite-state automaton (FSA) or pushdown automaton does:
track parsing state and define valid transitions.

For JSON, the state machine tracks things like:
- Are we inside a string? (only string characters and `"` are valid)
- Are we starting a new object key? (only `"` is valid)
- Did we just finish a value? (only `,`, `}`, `]` are valid)

### The Ecosystem

Structured decoding exploded in 2023–2024:

- **Outlines** (dottxt-ai): The research-grade library that formalized the FSA approach. Converts Pydantic schemas, JSON schemas, and regex patterns into token masks. Used internally by vLLM and TGI.
- **Instructor**: Higher-level library built on top of Pydantic + Outlines/prompt-based validation. Extremely popular for practical use.
- **guidance** (Microsoft): Interleaves generation and constraint enforcement in a single template language.
- **TGI (Text Generation Inference)**: Hugging Face's production serving framework includes built-in structured generation (`--grammar`).
- **vLLM**: Supports `guided_json`, `guided_regex`, `guided_choice` parameters natively.
- **llama.cpp / Ollama**: Support GBNF grammars (a BNF-style grammar format) for constrained generation.

---

## What You'll Build

1. **Token masking** — the core primitive: logit masking for constrained sampling
2. **Regex-constrained decoding** — enforce output matches a pattern
3. **JSON schema validation** — generate valid JSON with a fixed schema
4. **GBNF grammars** — the grammar format used by llama.cpp/Ollama
5. **Performance cost** — what structured decoding costs and how to minimize it
6. **Practical patterns** — when to use each approach

In [ ]:
!pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from dataclasses import dataclass, field
from typing import Optional, Callable
from enum import Enum

torch.manual_seed(42)
np.random.seed(42)

print("Setup complete.")

## Part 1: Token Masking — The Core Primitive

The fundamental operation in structured decoding is **logit masking**: before sampling,
set logits for invalid tokens to -infinity so they have zero probability after softmax.

In [ ]:
def apply_token_mask(
    logits: torch.Tensor,        # [vocab_size]
    allowed_token_ids: set[int], # which tokens are valid here
    mask_value: float = float('-inf'),
) -> torch.Tensor:
    """
    Mask out all tokens except those in allowed_token_ids.
    Returns new logits where invalid tokens have -inf logit.
    """
    masked = logits.clone()
    mask = torch.ones(len(logits), dtype=torch.bool)
    for tok_id in allowed_token_ids:
        if tok_id < len(logits):
            mask[tok_id] = False  # don't mask this one
    masked[mask] = mask_value
    return masked


def sample_with_mask(
    logits: torch.Tensor,
    allowed_token_ids: set[int],
    temperature: float = 1.0,
) -> int:
    """Sample the next token, constrained to allowed tokens."""
    masked_logits = apply_token_mask(logits, allowed_token_ids)
    if temperature != 1.0:
        masked_logits = masked_logits / temperature
    probs = F.softmax(masked_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# Demonstrate: suppose we have a vocabulary of 10 tokens
# and we want to constrain output to only digits [0-9]
VOCAB_SIZE = 10
DIGIT_TOKENS = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}  # all tokens are digits here
JSON_BRACE_TOKENS = {7}  # only '{' is valid at the start of a JSON object

# Simulate some logits (model's raw output)
logits = torch.tensor([0.1, 0.5, 0.2, -0.3, 0.8, -0.1, 0.4, 1.2, 0.3, -0.2])

print("Without mask:")
probs_no_mask = F.softmax(logits, dim=-1)
print(f"  Most likely token: {probs_no_mask.argmax().item()} (p={probs_no_mask.max():.3f})")

print("\nWith mask (only token 7 allowed — '{'):")
masked_logits = apply_token_mask(logits, JSON_BRACE_TOKENS)
probs_masked = F.softmax(masked_logits, dim=-1)
print(f"  Only allowed token: 7 (p={probs_masked[7]:.3f})")
print(f"  Other tokens: {probs_masked[:7].tolist()} (all 0)")

# Visualize the masking effect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vocab_labels = ['{', '}', '"', ':', ',', '[', ']', 'a', '0', '\\n']

ax = axes[0]
bars = ax.bar(range(VOCAB_SIZE), F.softmax(logits, dim=-1).numpy(),
              color='#3498db', alpha=0.8)
ax.set_title("Unconstrained (full vocabulary)")
ax.set_xticks(range(VOCAB_SIZE))
ax.set_xticklabels(vocab_labels)
ax.set_ylabel("Probability")

ax2 = axes[1]
allowed = {0, 1, 2, 3, 4, 5, 6}  # only JSON structure tokens
masked = apply_token_mask(logits, allowed)
probs_masked2 = F.softmax(masked, dim=-1).numpy()
colors2 = ['#2ecc71' if i in allowed else '#e74c3c' for i in range(VOCAB_SIZE)]
ax2.bar(range(VOCAB_SIZE), probs_masked2, color=colors2, alpha=0.8)
ax2.set_title("Masked (only JSON structure tokens allowed)")
ax2.set_xticks(range(VOCAB_SIZE))
ax2.set_xticklabels(vocab_labels)
ax2.set_ylabel("Probability")
legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Allowed'),
    mpatches.Patch(color='#e74c3c', label='Masked (p=0)'),
]
import matplotlib.patches as mpatches
ax2.legend(handles=legend_patches)

plt.suptitle("Token Masking: How Structured Decoding Works", fontsize=13)
plt.tight_layout()
plt.savefig('token_masking.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 2: Regex-Constrained Decoding

One of the most powerful applications of structured decoding: generate text that **exactly
matches a regex pattern**. This uses an incremental NFA (non-deterministic finite automaton)
that tracks which positions in the regex are currently active, and computes which characters
can advance the NFA without causing it to fail.

In [ ]:
class RegexConstraint:
    """
    Regex-based token constraint using Python's re module for incremental matching.
    
    For each generation step:
    1. Look at all possible single-character continuations
    2. Try each continuation: does the partial output + continuation still potentially match?
    3. Allow only characters that keep the match alive
    
    Note: This is a simplified version. Production implementations (like Outlines) compile
    the regex into a FSA and then precompute token masks for every possible FSA state —
    making inference-time lookups O(1) instead of O(vocab_size).
    """
    def __init__(self, pattern: str):
        self.pattern = pattern
        # Anchor the pattern for full-string matching
        self.full_pattern = re.compile('^' + pattern + '$')
        self.partial_pattern = re.compile('^' + pattern)
    
    def get_allowed_chars(self, partial_output: str) -> set[str]:
        """
        Given what's been generated so far, return the set of next characters
        that can lead to a valid completion.
        """
        # Check if already complete
        if self.full_pattern.match(partial_output):
            return set()  # Done! No more characters needed (but EOS is fine)
        
        allowed = set()
        # Try all printable ASCII characters
        for char in ' abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,!?-_/:\"\'\'{}[]()@#$%^&*+=<>\\|;`~\n\t':
            candidate = partial_output + char
            # Does any extension of `candidate` potentially match the full pattern?
            if self.partial_pattern.match(candidate):
                allowed.add(char)
        
        return allowed
    
    def is_complete(self, text: str) -> bool:
        return bool(self.full_pattern.match(text))
    
    def is_still_valid(self, partial: str) -> bool:
        """Can this partial output still become a valid match?"""
        return bool(self.partial_pattern.match(partial))


# Test with different patterns
print("Regex Constraint Examples:")
print("=" * 60)

test_cases = [
    (r'[0-9]{4}-[0-9]{2}-[0-9]{2}', '2024-01-', 'Date format after entering year-month'),
    (r'(yes|no)',                     '',           'Boolean yes/no choice'),
    (r'(yes|no)',                     'y',          'After entering y'),
    (r'[A-Z][a-z]+ [A-Z][a-z]+',     'John ',      'First Last name after first name'),
    (r'\$[0-9]+\.[0-9]{2}',          '$42.',        'Dollar amount after $42.'),
]

for pattern, partial, description in test_cases:
    constraint = RegexConstraint(pattern)
    allowed = constraint.get_allowed_chars(partial)
    complete = constraint.is_complete(partial)
    
    allowed_str = repr(sorted(allowed)[:10])
    if len(allowed) > 10:
        allowed_str = allowed_str[:-1] + f', ... ({len(allowed)} total)]'
    
    print(f"Pattern: {pattern!r}")
    print(f"  Partial: {partial!r}")
    print(f"  Description: {description}")
    print(f"  Allowed next chars: {allowed_str}")
    print(f"  Is complete: {complete}")
    print()

In [ ]:
# Character-level constrained generation demo
# We simulate token-by-token generation with a regex constraint

class ToyCharTokenizer:
    """A character-level tokenizer for demo purposes (one token = one character)."""
    CHARS = list(' abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,-:!/\'"{}[]()_@#$%^&+=<>;\\|`~\n')
    
    def __init__(self):
        self.vocab = {c: i for i, c in enumerate(self.CHARS)}
        self.inv_vocab = {i: c for c, i in self.vocab.items()}
        self.vocab_size = len(self.CHARS)
        self.eos_id = self.vocab_size  # special EOS token
    
    def char_to_id(self, c: str) -> Optional[int]:
        return self.vocab.get(c)
    
    def id_to_char(self, i: int) -> str:
        return self.inv_vocab.get(i, '?')


def constrained_generate(
    constraint: RegexConstraint,
    tokenizer: ToyCharTokenizer,
    max_tokens: int = 50,
    temperature: float = 0.8,
) -> str:
    """Generate text constrained to match the regex pattern."""
    generated = ''
    
    for step in range(max_tokens):
        if constraint.is_complete(generated):
            break
        
        # Get allowed chars at this position
        allowed_chars = constraint.get_allowed_chars(generated)
        if not allowed_chars:
            break
        
        # Convert to token IDs
        allowed_ids = {tokenizer.char_to_id(c) for c in allowed_chars if tokenizer.char_to_id(c) is not None}
        
        # Sample random logits (simulating model output)
        logits = torch.randn(tokenizer.vocab_size)
        
        # Apply mask and sample
        token_id = sample_with_mask(logits, allowed_ids, temperature)
        char = tokenizer.id_to_char(token_id)
        generated += char
    
    return generated


tok = ToyCharTokenizer()

# Generate dates
date_constraint = RegexConstraint(r'20[0-9]{2}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12][0-9]|3[01])')
print("Generated dates (constrained to valid date format):")
for _ in range(5):
    date = constrained_generate(date_constraint, tok, temperature=1.5)
    valid = date_constraint.is_complete(date)
    print(f"  {date!r}  ({'valid' if valid else 'INVALID'})")

print()

# Generate yes/no
yn_constraint = RegexConstraint(r'(yes|no)')
print("Generated yes/no answers:")
for _ in range(8):
    answer = constrained_generate(yn_constraint, tok, temperature=0.5)
    print(f"  {answer!r}", end='  ')
print()

## Part 3: JSON Schema Decoding

The most common use case: generate JSON that matches a specific schema.
We'll build a JSON state machine that tracks the parser state and
emits valid token masks at each position.

In [ ]:
class JSONState(Enum):
    """Parser states for JSON generation."""
    START          = 'start'
    OBJECT_OPEN    = 'object_open'    # just saw '{'
    EXPECT_KEY     = 'expect_key'     # expecting a string key
    IN_KEY         = 'in_key'         # inside a key string
    AFTER_KEY      = 'after_key'      # after closing '"' of key
    EXPECT_COLON   = 'expect_colon'   # expecting ':'
    EXPECT_VALUE   = 'expect_value'   # expecting a value
    IN_STRING      = 'in_string'      # inside a string value
    IN_NUMBER      = 'in_number'      # inside a number value
    AFTER_VALUE    = 'after_value'    # after completing a value
    DONE           = 'done'           # complete


class JSONSchemaConstraint:
    """
    State machine for generating JSON that matches a given schema.
    
    Supports a subset of JSON schema: object with string/number/boolean fields.
    """
    def __init__(self, schema: dict):
        self.schema = schema
        self.required_keys = list(schema.get('properties', {}).keys())
    
    def get_allowed_chars(self, partial: str) -> set[str]:
        """Compute allowed next characters given partial JSON output."""
        state, context = self._parse_state(partial)
        return self._allowed_from_state(state, context)
    
    def _parse_state(self, partial: str) -> tuple[JSONState, dict]:
        """Parse the partial JSON string to determine current state."""
        if not partial:
            return JSONState.START, {}
        
        try:
            # Try to parse as complete JSON
            json.loads(partial)
            return JSONState.DONE, {}
        except json.JSONDecodeError:
            pass
        
        # Incremental state parsing (simplified)
        stripped = partial.strip()
        
        if not stripped:
            return JSONState.START, {}
        if stripped == '{':
            return JSONState.OBJECT_OPEN, {}
        if stripped.endswith('{'):
            return JSONState.EXPECT_KEY, {}
        
        # Count quotes to determine string state
        n_quotes = stripped.count('"')
        
        # After key-colon, in value
        if ':' in stripped and not stripped.endswith('"') and n_quotes % 2 == 0:
            # Determine type from schema
            last_key = self._extract_last_key(stripped)
            field_type = self.schema.get('properties', {}).get(last_key, {}).get('type', 'string')
            return JSONState.IN_NUMBER if field_type == 'number' else JSONState.IN_STRING, {'key': last_key}
        
        if stripped.endswith(':'):
            return JSONState.EXPECT_VALUE, {}
        
        # Default: in string or number
        if n_quotes % 2 == 1:
            return JSONState.IN_STRING, {}
        
        return JSONState.AFTER_VALUE, {}
    
    def _extract_last_key(self, partial: str) -> str:
        """Extract the most recent key from partial JSON."""
        keys = re.findall(r'"([^"]+)"\s*:', partial)
        return keys[-1] if keys else ''
    
    def _allowed_from_state(self, state: JSONState, context: dict) -> set[str]:
        """Return allowed characters for each parser state."""
        if state == JSONState.START:
            return {'{'}
        elif state == JSONState.OBJECT_OPEN or state == JSONState.EXPECT_KEY:
            return {'"'}
        elif state == JSONState.IN_KEY:
            # Allow alphanumeric and common key chars, plus closing quote
            return set('abcdefghijklmnopqrstuvwxyz_ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"')
        elif state == JSONState.AFTER_KEY or state == JSONState.EXPECT_COLON:
            return {':'}
        elif state == JSONState.EXPECT_VALUE:
            field_type = context.get('type', 'string')
            if field_type == 'number':
                return set('0123456789-')
            elif field_type == 'boolean':
                return {'t', 'f'}  # true or false
            else:
                return {'"'}
        elif state == JSONState.IN_STRING:
            return set('abcdefghijklmnopqrstuvwxyz ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,!?-_/\'()@# "')
        elif state == JSONState.IN_NUMBER:
            return set('0123456789.')
        elif state == JSONState.AFTER_VALUE:
            return {',', '}'}
        elif state == JSONState.DONE:
            return set()
        return set()


# Define a schema
person_schema = {
    'type': 'object',
    'properties': {
        'name': {'type': 'string'},
        'age': {'type': 'number'},
        'city': {'type': 'string'},
    },
    'required': ['name', 'age', 'city'],
}

constraint = JSONSchemaConstraint(person_schema)

# Show what's allowed at each step of a JSON generation
partial_states = [
    '',
    '{',
    '{"name"',
    '{"name":',
    '{"name": "Alice',
    '{"name": "Alice", ',
    '{"name": "Alice", "age":',
    '{"name": "Alice", "age": 30',
]

print("JSON generation state machine:")
print(f"{'Partial output':<40} {'Allowed next chars'}")
print("-" * 80)
for partial in partial_states:
    allowed = constraint.get_allowed_chars(partial)
    allowed_str = repr(sorted(allowed)[:8])
    if len(allowed) > 8:
        allowed_str = allowed_str[:-1] + f', ... ({len(allowed)} total)]'
    print(f"{partial!r:<40} {allowed_str}")

## Part 4: GBNF Grammars (llama.cpp format)

llama.cpp and Ollama use GBNF (a BNF-like grammar format) for structured generation.
GBNF is more powerful than JSON schema alone — it can express any context-free grammar.

In [ ]:
# GBNF examples

json_grammar = '''
# GBNF grammar for simple JSON objects
# This is what llama.cpp/Ollama uses internally

root   ::= object
object ::= "{" ws (string ":" ws value ("," ws string ":" ws value)*)? "}"
array  ::= "[" ws (value ("," ws value)*)? "]"
value  ::= object | array | string | number | ("true" | "false" | "null")
string ::= "\"" (([^\"\\\x7F\x00-\x1F] | \"\\\" ["\\/bfnrt] | "\\\" "u" [0-9a-fA-F] {4}))* "\""
number ::= "-"? ([0-9] | [1-9] [0-9]*) ("." [0-9]+)? (([eE] [-+]? [0-9]+))?
ws     ::= ([ \t\n] ws)?
'''

sentiment_grammar = '''
# Constrain output to sentiment classification
root ::= "positive" | "negative" | "neutral"
'''

structured_review = '''
# A specific structured output format
root     ::= "{" ws "\"sentiment\"" ws ":" ws sentiment "," ws "\"score\"" ws ":" ws score ws "}"
sentiment ::= "\"positive\"" | "\"negative\"" | "\"neutral\""
score    ::= [0-9] "." [0-9]
ws       ::= " "?
'''

print("GBNF Grammar Examples:")
print("=" * 70)
print("1. JSON Object Grammar:")
print(json_grammar)
print("2. Sentiment Classification:")
print(sentiment_grammar)
print("3. Structured Review Output:")
print(structured_review)

# How to use with Ollama
print("Using GBNF with Ollama API:")
print("-" * 50)
ollama_example = '''
import requests

grammar = '''
root ::= \'\'{' ws sentiment ws '}\'\''
'''

response = requests.post(
    'http://localhost:11434/api/generate',
    json={
        'model': 'llama3.1:8b',
        'prompt': 'Classify the sentiment: I love this product!',
        'format': 'json',     # Use format='json' for JSON mode
        # OR use grammar=... for custom GBNF grammar
        'stream': False,
    }
)
output = response.json()['response']
parsed = json.loads(output)  # Guaranteed to parse!
'''
print(ollama_example)

## Part 5: The Performance Cost of Structured Decoding

Structured decoding has a latency cost. The mask must be computed at every token step.
How expensive is it, and how do libraries minimize the overhead?

In [ ]:
import time

def benchmark_mask_computation(vocab_size: int, n_steps: int = 1000) -> dict:
    """Benchmark different mask computation approaches."""
    results = {}
    
    # Approach 1: Naive (compute mask from scratch each step)
    start = time.perf_counter()
    for _ in range(n_steps):
        allowed = set(range(0, vocab_size, 7))  # ~14% of vocab allowed
        logits = torch.randn(vocab_size)
        _ = apply_token_mask(logits, allowed)
    results['naive_ms'] = (time.perf_counter() - start) * 1000 / n_steps
    
    # Approach 2: Pre-computed boolean mask tensor (much faster)
    precomputed_mask = torch.ones(vocab_size, dtype=torch.bool)
    for i in range(0, vocab_size, 7):
        precomputed_mask[i] = False
    
    start = time.perf_counter()
    for _ in range(n_steps):
        logits = torch.randn(vocab_size)
        masked = logits.clone()
        masked[precomputed_mask] = float('-inf')
    results['precomputed_ms'] = (time.perf_counter() - start) * 1000 / n_steps
    
    # Approach 3: Per-state cache (Outlines approach)
    # Precompute masks for all possible FSA states during model loading
    # During inference: just do a table lookup (O(1) per step)
    state_masks = {}  # state_id -> precomputed mask tensor
    for state_id in range(100):  # 100 possible FSA states
        mask = torch.ones(vocab_size, dtype=torch.bool)
        allowed_for_state = set(range(state_id, vocab_size, 100))
        for i in allowed_for_state:
            mask[i] = False
        state_masks[state_id] = mask
    
    start = time.perf_counter()
    for step in range(n_steps):
        state_id = step % 100
        logits = torch.randn(vocab_size)
        masked = logits.clone()
        masked[state_masks[state_id]] = float('-inf')
    results['cached_ms'] = (time.perf_counter() - start) * 1000 / n_steps
    
    return results


vocab_sizes = [32000, 128000, 200000]

print("Mask computation overhead per token step:")
print(f"{'Vocab Size':<12} {'Naive (ms)':>12} {'Precomputed (ms)':>18} {'Cached (ms)':>13}")
print("-" * 60)
for vocab_size in vocab_sizes:
    r = benchmark_mask_computation(vocab_size)
    print(f"{vocab_size:<12,} {r['naive_ms']:>12.3f} {r['precomputed_ms']:>18.3f} {r['cached_ms']:>13.3f}")

print("\nFor context: model decode step is ~5-50ms per token on typical hardware.")
print("Cached mask lookup is typically < 1% of total decode time — negligible.")
print("This is why Outlines precomputes all state masks during model loading.")

## Part 6: Using Structured Decoding in Practice

How to use structured decoding with real tools: Outlines, Instructor, and vLLM.

In [ ]:
# Code patterns for the main libraries

outlines_example = '''
# ─── Outlines: Pydantic schema → structured generation ───────────────────────
# pip install outlines
from pydantic import BaseModel
import outlines

class PersonInfo(BaseModel):
    name: str
    age: int
    occupation: str
    is_employed: bool

model = outlines.models.transformers("meta-llama/Llama-3.1-8B-Instruct")
generator = outlines.generate.json(model, PersonInfo)  # <-- that\'s all

result = generator("Generate a fictional person:")
print(result)  # Always a valid PersonInfo object
print(result.name, result.age)  # Type-safe attribute access
'''

instructor_example = '''
# ─── Instructor: structured extraction from any LLM API ─────────────────────
# pip install instructor
import instructor
from openai import OpenAI
from pydantic import BaseModel

client = instructor.from_openai(OpenAI())
# Also: instructor.from_anthropic(), instructor.from_ollama()

class MovieReview(BaseModel):
    title: str
    year: int
    sentiment: Literal["positive", "negative", "mixed"]
    score: float  # 0.0 to 10.0
    summary: str

review = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=MovieReview,
    messages=[{"role": "user", "content": "Review: Inception is a mind-bending thriller..."}],
)
print(review.sentiment, review.score)  # Guaranteed valid types
'''

vllm_guided_example = '''
# ─── vLLM: built-in structured generation ────────────────────────────────────
from vllm import LLM, SamplingParams
from pydantic import BaseModel

class ProductAnalysis(BaseModel):
    product_name: str
    category: str
    price_range: str
    sentiment: str

llm = LLM(model="meta-llama/Llama-3.1-8B-Instruct")

sampling_params = SamplingParams(
    temperature=0.7,
    guided_json=ProductAnalysis.model_json_schema(),  # <-- vLLM handles masking
    # OR: guided_regex=r"(positive|negative|neutral)"
    # OR: guided_choice=["yes", "no", "maybe"]
)

outputs = llm.generate(["Analyze this product: iPhone 15 Pro..."], sampling_params)
import json
result = ProductAnalysis(**json.loads(outputs[0].outputs[0].text))
'''

for label, example in [
    ('Outlines', outlines_example),
    ('Instructor', instructor_example),
    ('vLLM Guided', vllm_guided_example),
]:
    print(f"{'=' * 70}")
    print(f"  {label}")
    print(f"{'=' * 70}")
    print(example)

## Part 7: When to Use Structured Decoding

Structured decoding is powerful but not always the right tool.

In [ ]:
use_cases = [
    # (use case, approach, reason)
    (
        "Extract structured data from text",
        "✅ Instructor + Pydantic",
        "Schema enforces field types; model fills in values from context"
    ),
    (
        "Classification (sentiment, category)",
        "✅ Guided choice or regex",
        "Constrain to exact label set; no hallucinated classes"
    ),
    (
        "Generate configuration files (JSON/YAML)",
        "✅ JSON schema or GBNF grammar",
        "Guaranteed valid syntax; schema validates field names"
    ),
    (
        "Fill out forms or templates",
        "✅ Pydantic model",
        "Type safety + validation rules (min/max, regex patterns)"
    ),
    (
        "Code generation",
        "⚠️ Sometimes useful",
        "Grammar can enforce syntactically valid code, but semantics still unconstrained"
    ),
    (
        "Open-ended creative writing",
        "❌ Don't use",
        "Constraints hurt quality for free-form generation"
    ),
    (
        "Chain-of-thought reasoning",
        "⚠️ Use carefully",
        "Constraining CoT steps can block the model's reasoning process"
    ),
    (
        "Tool call arguments",
        "✅ Strong recommendation",
        "Tool arguments must be structurally correct; JSON schema guarantees this"
    ),
    (
        "Agent decision outputs",
        "✅ Pydantic + enum for action type",
        "Ensures agent always picks a valid action from defined action space"
    ),
]

print("When to use structured decoding:")
print("=" * 90)
print(f"{'Use Case':<40} {'Approach':<25} Reason")
print("-" * 90)
for use_case, approach, reason in use_cases:
    print(f"{use_case:<40} {approach:<25} {reason}")

## Summary

Structured decoding transforms LLMs from unreliable text generators into reliable structured
data processors — without sacrificing quality.

### Key Takeaways

**The Core Mechanism**  
At every decode step, compute a mask over the vocabulary that zeroes out all tokens that
would violate the output schema. Sample from the remaining (valid) distribution. Output
is guaranteed structurally valid — 100%, not "usually."

**The Three Approaches**  
- **Token masking + FSA**: The fundamental technique. Outlines compiles schemas into finite-state
  automata and precomputes masks for each state (O(1) per decode step).
- **GBNF grammars**: llama.cpp/Ollama's BNF-style format. Expressive enough for any context-free grammar.
- **JSON schema / Pydantic**: The practical API. vLLM, Instructor, and Outlines all accept Pydantic models.

**Performance**  
With precomputed state masks, the overhead is < 1% of decode time. The Outlines approach
of compiling grammars at load time (not inference time) is the key insight.

**When to Use It**  
Anywhere you need type-safe, schema-valid output: data extraction, classification, tool calls,
agent action spaces, configuration generation. Don't use it for free-form creative tasks.

### What's Next (Notebook 16: Positional Encoding & RoPE)

We've covered all the major inference efficiency techniques. Notebook 16 shifts to a
fundamental model property: how transformers handle position. RoPE (Rotary Positional
Embedding) is used in every major modern model (LLaMA, Mistral, Gemma, Qwen) and its
properties directly determine how well a model generalizes to longer contexts than it was trained on.